# Assessment Notebook

In [1]:
# Numerical arrays
import numpy as np

## Problem 1: Representing SHA-256 Data

SHA-256 uses different types of data when processing a message and producing a hash. These include the original message, 32-bit words, 512-bit message blocks and the final 256-bit hash value.

Python provides different ways to store and work with these values. For example, `bytes` can be used for the original message, while NumPy provides `np.uint32` for the 32-bit words used in the SHA-256 calculations.

In this problem, I will demonstrate how each type of data can be represented in Python. I will also explain why I have chosen each representation and show small examples of how they work.

The main reference for this problem is the NIST Secure Hash Standard (FIPS 180-4). I will use Sections 3.1, 5.1.1, 5.2.1 and 6.2.2 to understand the data sizes, message format and SHA-256 calculations.

Reference:
https://doi.org/10.6028/NIST.FIPS.180-4

#### 32-bit words

A 32-bit word contains exactly 32 bits and can represent an unsigned integer between 0 and 4,294,967,295. SHA-256 uses 32-bit words throughout its calculations, so it is important that these values are represented correctly.

Python integers can store numbers larger than 32 bits, so they do not automatically have the same size as the words used by SHA-256. NumPy provides `np.uint32`, which represents an unsigned 32-bit integer. I have used this in my example to make sure the word uses the correct data type.

I also looked at a Stack Overflow discussion about fixed-width integers in Python. It explains why using a fixed-size data type can be useful when performing calculations that depend on the number of bits.

References:

- NIST FIPS 180-4, Section 3.1 – Bit Strings and Integers:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – Data types:
  https://numpy.org/doc/stable/user/basics.types.html

- Stack Overflow – Fixed-width integer types in Python:
  https://stackoverflow.com/questions/31145143/fixed-width-integer-types-e-g-uint32-in-python

In [2]:
# Represent a 32-bit word
word = np.uint32(0x12345678)

print("Word:", hex(int(word)))
print("Data type:", word.dtype)

assert word.dtype == np.uint32
assert int(word) == 0x12345678

print("Test passed!")

Word: 0x12345678
Data type: uint32
Test passed!


#### Sequences of 32-bit words

SHA-256 uses sequences of 32-bit words when processing messages and performing its calculations. For example, each 512-bit message block is divided into sixteen 32-bit words.

In Python, these sequences can be represented using a list or a NumPy array. I have chosen a NumPy array because it allows all the values to use the same data type. By setting the data type to `np.uint32`, each value is stored as an unsigned 32-bit integer.

I looked at the NumPy array creation documentation to understand how to create an array with a specific data type. This is useful because it allows me to create a sequence of words that can be accessed using their index.

References:

- NIST FIPS 180-4, Section 6.2.2 – SHA-256 Hash Computation:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – Array creation:
  https://numpy.org/doc/stable/user/basics.creation.html

- Stack Overflow – Fixed-width integer types in Python:
  https://stackoverflow.com/questions/31145143/fixed-width-integer-types-e-g-uint32-in-python

In [3]:
# Represent a sequence of 32-bit words
words = np.array(
    [0x12345678, 0xABCDEF01, 0x87654321],
    dtype=np.uint32
)

print([hex(int(word)) for word in words])

assert len(words) == 3
assert words.dtype == np.uint32

print("Test passed!")

['0x12345678', '0xabcdef01', '0x87654321']
Test passed!


#### Input messages

SHA-256 can process messages of different lengths. In Python, an input message can be represented using a `bytes` object because SHA-256 processes the actual bytes of the message rather than Python characters.

A normal Python string (`str`) must first be converted into bytes before it can be used as input. The `encode()` method can be used for this, and UTF-8 is a suitable choice because it supports normal ASCII text as well as non-ASCII characters.

For example, the string "abc" can be converted into a bytes object using UTF-8 encoding. The resulting bytes can then be used by the SHA-256 algorithm.

I also looked at a Stack Overflow discussion about encoding strings before using them with Python hashing functions. It helped explain why a normal string needs to be converted into bytes first.

References:

- NIST FIPS 180-4, Section 5.1.1 – SHA-1, SHA-224 and SHA-256 Padding:
  https://doi.org/10.6028/NIST.FIPS.180-4

- Python – str.encode():
  https://docs.python.org/3/library/stdtypes.html#str.encode

- Stack Overflow – Encoding strings before hashing:
  https://stackoverflow.com/questions/7585307/how-to-correct-typeerror-unicode-objects-must-be-encoded-before-hashing

In [4]:
# Represent an input message as bytes
message = "abc".encode("utf-8")

print(message)
print("Length:", len(message), "bytes")

assert message == b"abc"
assert isinstance(message, bytes)

print("Test passed!")

b'abc'
Length: 3 bytes
Test passed!


#### 512-bit message blocks

SHA-256 processes messages in blocks of 512 bits. Each block contains 64 bytes, which can also be divided into sixteen 32-bit words.

In Python, a block can be represented using a `bytes` object. NumPy can then be used to interpret these bytes as sixteen 32-bit unsigned integers.

I have used `np.frombuffer()` for this because it allows the bytes to be read directly as a NumPy array. I have also used `>u4` as the data type. This means that each value is an unsigned four-byte integer using big-endian byte order.

I looked at the NumPy documentation and a Stack Overflow discussion about converting bytes into 32-bit integers. Both explain how bytes can be interpreted as numerical values using `frombuffer()`. This is useful for understanding how I will convert message blocks into words in Problem 4.

The example below uses 64 bytes to demonstrate the structure of one complete block. The actual message padding and splitting into blocks will be implemented later.

References:

- NIST FIPS 180-4, Section 5.2.1 – SHA-1, SHA-224 and SHA-256 Message Parsing:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – numpy.frombuffer():
  https://numpy.org/doc/stable/reference/generated/numpy.frombuffer.html

- Stack Overflow – Converting bytes into 32-bit integer values:
  https://stackoverflow.com/questions/77887703/how-to-convert-a-big-bytes-object-to-a-list-of-32-bit-integer-values

In [5]:
# Represent a 512-bit block as sixteen 32-bit words
block = bytes(range(64))

block_words = np.frombuffer(
    block, dtype=">u4"
).astype(np.uint32)

print("Block size:", len(block) * 8, "bits")
print("Number of words:", len(block_words))

assert len(block_words) == 16
assert int(block_words[0]) == 0x00010203

print("Test passed!")

Block size: 512 bits
Number of words: 16
Test passed!


#### The final 256-bit hash value

SHA-256 always produces a hash value that is 256 bits long, regardless of the length of the original message. This means that the final hash contains 32 bytes.

In Python, the hash can be represented using a `bytes` object. It can also be converted into a hexadecimal string using the `hex()` method. A hexadecimal representation contains 64 characters because each character represents four bits.

I looked at Python's `hashlib` documentation to understand how SHA-256 produces its final digest. It explains the difference between `digest()`, which returns the hash as bytes, and `hexdigest()`, which returns it as a hexadecimal string.

In the example below, I use `hashlib` to generate an actual SHA-256 hash of the message "abc". I will use this again in Problem 6 when testing my own SHA-256 implementation.

References:

- NIST FIPS 180-4, Section 6.2.2 – SHA-256 Hash Computation:
  https://doi.org/10.6028/NIST.FIPS.180-4

- Python – hashlib:
  https://docs.python.org/3/library/hashlib.html

- Python – bytes.hex():
  https://docs.python.org/3/library/stdtypes.html#bytes.hex

In [6]:
# Generate and represent a SHA-256 hash
import hashlib

hash_value = hashlib.sha256(b"abc").digest()

print("Hash:", hash_value.hex())
print("Hash size:", len(hash_value) * 8, "bits")

assert len(hash_value) == 32
assert len(hash_value.hex()) == 64

print("Test passed!")

Hash: ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad
Hash size: 256 bits
Test passed!


#### Big-endian integers

Big-endian is a way of storing numbers where the most significant byte comes first. For example, the 32-bit integer `0x12345678` is stored as four bytes: `12 34 56 78`.

SHA-256 uses big-endian byte ordering when interpreting message blocks as 32-bit words. This means that the order of the bytes is important because interpreting them in the wrong order would produce a different integer.

Python provides `to_bytes()` to convert an integer into bytes and `int.from_bytes()` to convert bytes back into an integer. Both methods allow the byte order to be specified.

I also looked at a Stack Overflow discussion about endianness. It explains why the same bytes can produce different integer values depending on whether they are interpreted as big-endian or little-endian. This helped me understand why I need to specify the byte order when working with SHA-256.

References:

- NIST FIPS 180-4, Section 3.1 – Bit Strings and Integers:
  https://doi.org/10.6028/NIST.FIPS.180-4

- Python – int.to_bytes():
  https://docs.python.org/3/library/stdtypes.html#int.to_bytes

- Stack Overflow – Why do we need endianness here?:
  https://stackoverflow.com/questions/33692321/why-do-we-need-endianness-here

In [7]:
# Convert an integer into big-endian bytes
word = 0x12345678

big_endian = word.to_bytes(4, byteorder="big")
converted_word = int.from_bytes(big_endian, byteorder="big")

print("Bytes:", big_endian)
print("Integer:", hex(converted_word))

assert converted_word == word

print("Test passed!")

Bytes: b'\x124Vx'
Integer: 0x12345678
Test passed!


## Problem 2: SHA-256 Bitwise Operations

SHA-256 uses a number of Boolean and bitwise operations to process messages and calculate the final hash value. These operations work with 32-bit words and are an important part of the compression algorithm.

In this problem, I will implement the six functions defined in the Secure Hash Standard. These include the Choice and Majority functions, along with four functions that use rotations and right shifts.

I will use NumPy's `uint32` data type to represent the values because SHA-256 works with fixed-width 32-bit unsigned integers. This is important because Python integers can grow beyond 32 bits, whereas SHA-256 requires calculations to stay within this range.

There are two main operations used by these functions. A right rotation (ROTR) moves bits to the right and brings any bits that leave the right side back to the left side. A right shift (SHR) moves bits to the right and fills the empty positions with zeros.

Each function will be implemented separately and tested using example inputs to check that the results are correct.

References:

- NIST FIPS 180-4, Section 4.1.2 – Defines the six functions used by SHA-256.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – Python SHA-256 hash computation. Discusses implementing SHA-256 and handling values within the required bit range.
  https://stackoverflow.com/questions/55856773/python-sha256-hash-computation

### Ch(x, y, z)

The Choice function takes three 32-bit words called x, y and z. It chooses bits from y or z depending on the corresponding bit in x. When a bit in x is 1, the function selects the corresponding bit from y. When a bit in x is 0, it selects the corresponding bit from z.

The function is defined as:

Ch(x, y, z) = (x AND y) XOR (NOT x AND z)

In Python, `&` performs bitwise AND, `^` performs bitwise XOR and `~` performs bitwise NOT. These operators allow the function to work directly with individual bits rather than converting the values into binary strings.

This function is used during the SHA-256 compression process. It helps calculate the temporary values used to update the working variables in each round.

References:

- NIST FIPS 180-4, Section 4.1.2, Equation 4.2 – Defines the Choice function.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – Python SHA-256 hash computation. Provides a practical discussion of implementing SHA-256 operations in Python.
  https://stackoverflow.com/questions/55856773/python-sha256-hash-computation

In [8]:
# Choice function Ch(x, y, z)
def Ch(x, y, z):
    """Return the SHA-256 choice function."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (~x & z)


# Test the Choice function
result = Ch(0b1100, 0b1010, 0b0110)

print("Result:", bin(int(result)))

assert int(result) == 0b1010
print("Test passed!")

Result: 0b1010
Test passed!


### Maj(x, y, z)

The Majority function takes three 32-bit words called x, y and z. It checks each bit position and returns 1 when at least two of the three corresponding bits are 1. Otherwise, it returns 0.

The function is defined as:

Maj(x, y, z) = (x AND y) XOR (x AND z) XOR (y AND z)

The `&` operator performs bitwise AND, while `^` performs bitwise XOR. By combining these operations, the function calculates the majority value for each bit position.

This function is used during the compression process alongside the Choice function. It helps calculate one of the temporary values used to update the working variables.

References:

- NIST FIPS 180-4, Section 4.1.2, Equation 4.3 – Defines the Majority function.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – Python SHA-256 hash computation. Provides a practical example of implementing SHA-256 in Python.
  https://stackoverflow.com/questions/55856773/python-sha256-hash-computation

In [9]:
# Majority function Maj(x, y, z)
def Maj(x, y, z):
    """Return the SHA-256 majority function."""
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    return (x & y) ^ (x & z) ^ (y & z)


# Test the Majority function
result = Maj(0b1100, 0b1010, 0b0110)

print("Result:", bin(int(result)))

assert int(result) == 0b1110
print("Test passed!")

Result: 0b1110
Test passed!


### Sigma0(x)

The uppercase Sigma0 function takes one 32-bit word called x and performs three right rotations. It rotates the bits by 2, 13 and 22 positions, then combines the results using bitwise XOR.

The function is defined as:

Sigma0(x) = ROTR2(x) XOR ROTR13(x) XOR ROTR22(x)

A right rotation moves bits to the right, but any bits that move past the end are brought back to the left side. This is different from a normal right shift, which moves bits to the right and fills the empty positions with zeros.

In Python, `>>` and `<<` are used to perform the shifts. The `|` operator combines the shifted values to create each rotation, while `^` combines the three rotations.

This function is used during the compression process to help calculate the temporary values needed to update the working variables.

References:

- NIST FIPS 180-4, Section 4.1.2, Equation 4.4 – Defines the uppercase Sigma0 function.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – Bitwise Rotate Right. Explains how left and right shifts can be combined to perform a bit rotation.
  https://stackoverflow.com/questions/27176317/bitwise-rotate-right

In [10]:
# Sigma0 function
def Sigma0(x):
    """Return the SHA-256 uppercase Sigma0 function."""
    x = np.uint32(x)
    return (
        ((x >> 2) | (x << 30)) ^
        ((x >> 13) | (x << 19)) ^
        ((x >> 22) | (x << 10))
    )


# Test the Sigma0 function
result = Sigma0(0b1100)

print("Result:", int(result))

assert int(result) == 6303747
assert result.dtype == np.uint32

print("Test passed!")

Result: 6303747
Test passed!


### Sigma1(x)

The uppercase Sigma1 function works in a similar way to Sigma0. It takes one 32-bit word and performs three right rotations. However, it uses different rotation amounts: 6, 11 and 25 positions.

The function is defined as:

Sigma1(x) = ROTR6(x) XOR ROTR11(x) XOR ROTR25(x)

Each rotation moves the bits to the right and brings the bits that leave the right side back to the left side. The three rotated values are then combined using bitwise XOR.

This function is also used during the compression process. It helps calculate the temporary values used to update the working variables in each round.

References:

- NIST FIPS 180-4, Section 4.1.2, Equation 4.5 – Defines the uppercase Sigma1 function.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – Bitwise Rotate Right. Explains the rotation technique used in this function.
  https://stackoverflow.com/questions/27176317/bitwise-rotate-right

In [11]:
# Sigma1 function
def Sigma1(x):
    """Return the SHA-256 uppercase Sigma1 function."""
    x = np.uint32(x)
    return (
        ((x >> 6) | (x << 26)) ^
        ((x >> 11) | (x << 21)) ^
        ((x >> 25) | (x << 7))
    )


# Test the Sigma1 function
result = Sigma1(0b1100)

print("Result:", int(result))

assert int(result) == 830473728
assert result.dtype == np.uint32

print("Test passed!")

Result: 830473728
Test passed!


### sigma0(x)

The lowercase sigma0 function takes one 32-bit word and performs three operations on its bits. Unlike the uppercase Sigma functions, it uses two right rotations and one normal right shift. The rotation amounts are 7 and 18 positions, while the normal right shift is 3 positions.

The function is defined as:

sigma0(x) = ROTR7(x) XOR ROTR18(x) XOR SHR3(x)

The main difference between a rotation and a normal right shift is that a rotation brings the bits back around to the opposite side, while a normal right shift moves the bits to the right and fills the empty positions with zeros.

This function is used when SHA-256 creates the message schedule. It helps calculate new 32-bit words from earlier words in the message block.

References:

- NIST FIPS 180-4, Section 4.1.2, Equation 4.6 – Defines the lowercase sigma0 function.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – SHA-256 sigma0 function. Discusses the rotation and shift operations used in sigma0.
  https://stackoverflow.com/questions/66607696/reverse-sha-256-sigma0-function-within-complexity-of-on

In [12]:
# sigma0 function
def sigma0(x):
    """Return the SHA-256 lowercase sigma0 function."""
    x = np.uint32(x)
    return (
        ((x >> 7) | (x << 25)) ^
        ((x >> 18) | (x << 14)) ^
        (x >> 3)
    )


# Test the sigma0 function
result = sigma0(0b1100)

print("Result:", int(result))

assert int(result) == 402849793
assert result.dtype == np.uint32

print("Test passed!")

Result: 402849793
Test passed!


### sigma1(x)

The lowercase sigma1 function works in a similar way to lowercase sigma0. It takes one 32-bit word and performs two right rotations and one normal right shift.

The rotation amounts are 17 and 19 positions, while the normal right shift is 10 positions.

The function is defined as:

sigma1(x) = ROTR17(x) XOR ROTR19(x) XOR SHR10(x)

The two rotations bring bits that leave the right side back to the left side. The normal right shift moves the bits to the right and fills the empty positions with zeros.

This function is also used when SHA-256 creates the message schedule. Together with sigma0, it helps generate the remaining words needed for the 64 rounds of the compression algorithm.

References:

- NIST FIPS 180-4, Section 4.1.2, Equation 4.7 – Defines the lowercase sigma1 function.
  https://doi.org/10.6028/NIST.FIPS.180-4

- Stack Overflow – SHA-256 implementation in Python. Discusses the implementation of SHA-256 and its smaller sigma functions.
  https://stackoverflow.com/questions/7321694/sha-256-implementation-in-python

In [13]:
# sigma1 function
def sigma1(x):
    """Return the SHA-256 lowercase sigma1 function."""
    x = np.uint32(x)
    return (
        ((x >> 17) | (x << 15)) ^
        ((x >> 19) | (x << 13)) ^
        (x >> 10)
    )


# Test the sigma1 function
result = sigma1(0b1100)

print("Result:", int(result))

assert int(result) == 491520
assert result.dtype == np.uint32

print("Test passed!")

Result: 491520
Test passed!


## Problem 3: Generating the SHA-256 Constants

SHA-256 uses 64 constants during its compression process. These constants are generated using the fractional parts of the cube roots of the first 64 prime numbers.

In this problem, I will use Python and NumPy to generate these constants. I will start by generating the first 64 prime numbers, then calculate their cube roots and extract the fractional parts. Finally, I will convert the results into 32-bit unsigned integers and compare them with the official values from the Secure Hash Standard.

The main reference for this problem is the NIST Secure Hash Standard (FIPS 180-4), particularly Section 4.2.2, which explains how the SHA-256 constants are generated.

Reference:

- NIST FIPS 180-4, Section 4.2.2 – SHA-256 Constants:
  https://doi.org/10.6028/NIST.FIPS.180-4

### Generating prime numbers

A prime number is a number greater than 1 that can only be divided evenly by 1 and itself. SHA-256 uses the first 64 prime numbers to generate its round constants.

I have created a function called `primes(n)` to generate the first `n` prime numbers. It checks whether each number can be divided evenly by any number up to its square root. If it cannot, it is added to the list.

I have used a NumPy array to store the results because I will need these numbers for the calculations in the following steps.

References:

- NIST FIPS 180-4, Section 4.2.2:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – Array creation:
  https://numpy.org/doc/stable/user/basics.creation.html

- Stack Overflow – Generating prime numbers in Python:
  https://stackoverflow.com/questions/2068372/fastest-way-to-list-all-primes-below-n

In [14]:
# Generate the first n prime numbers
def primes(n):
    prime_numbers = []
    num = 2

    while len(prime_numbers) < n:
        for i in range(2, int(np.sqrt(num)) + 1):
            if num % i == 0:
                break
        else:
            prime_numbers.append(num)

        num += 1

    return np.array(prime_numbers, dtype=np.uint32)


# Generate the first 64 primes
p = primes(64)

print(p)

assert len(p) == 64
assert p[-1] == 311

print("Test passed!")

[  2   3   5   7  11  13  17  19  23  29  31  37  41  43  47  53  59  61
  67  71  73  79  83  89  97 101 103 107 109 113 127 131 137 139 149 151
 157 163 167 173 179 181 191 193 197 199 211 223 227 229 233 239 241 251
 257 263 269 271 277 281 283 293 307 311]
Test passed!


### Calculating the cube roots

Once I have the prime numbers, the next step is to calculate their cube roots.

SHA-256 uses the fractional parts of these cube roots to generate its constants. I have used NumPy's `np.cbrt()` function to calculate them. This allows me to perform the calculation on the entire array rather than calculating each value separately.

The results contain both whole numbers and decimal parts. I will separate these in the next step.

References:

- NIST FIPS 180-4, Section 4.2.2:
  https://doi.org/10.6028/NIST.FIPS.180-4

- NumPy – numpy.cbrt():
  https://numpy.org/doc/stable/reference/generated/numpy.cbrt.html

In [15]:
# Calculate the cube roots of the prime numbers
cube_roots = np.cbrt(p)

print(cube_roots[:5])

assert len(cube_roots) == 64

print("Test passed!")

[1.25992105 1.44224957 1.70997595 1.91293118 2.22398009]
Test passed!


## Problem 4: Padding and Parsing Messages

Problem 4 involves write a generator function called `block_parse(msg)` that accepts a `bytes` object and yields the 512-bit message blocks required by SHA-256.

Each yielded block should contain sixteen 32-bit unsigned integers.

Implement the padding specified in sections 5.1.1 and 5.2.1 of the Secure Hash Standard, including:

- appending the `1` bit,

- appending the required zero bits,

- appending the original message length as a 64-bit integer,

- handling messages that require one or two final blocks,

- interpreting the message using the required byte order.

Test your implementation, including any boundary cases you can identify.

## Problem 5: The SHA-256 Compression Function

Problem 5 involves implementing the SHA-256 compression algorithm described in section 6.2.2 of the Secure Hash Standard.

Your implementation should explicitly construct the 64-word message schedule and perform the 64 rounds of the compression function.

Structure your implementation so that the major stages of the algorithm are identifiable in your code.

Explain:

- how the message schedule is constructed;
- the purpose of the eight working variables;
- how `Ch` and `Maj` are used;
- how the two Σ functions are used;
- how the round constants are incorporated;
- how the working variables are updated;
- how the compressed state is combined with the previous hash state.

Use your `block_parse` generator and the functions implemented in the previous problems.

## Problem 6: Complete SHA-256

Problem 6 involves combining your previous work into a function:

> sha256(msg)

The function should accept a `bytes` object and return its SHA-256 digest as a 64-character lowercase hexadecimal string.

Test your implementation against Python's `hashlib.sha256` using a range of inputs, including:

- the empty message,
- short ASCII strings,
- messages containing non-ASCII characters,
- messages whose lengths are close to padding boundaries,
- a longer message.